# CRISPR-Cas9 off-target/activity ML extension

This notebook adapts FlyRank notebook 04's sequence features, grouped split, leakage audit, and uncertainty reporting to a public SITE-seq benchmark. It produces a review ranking; it does not replace genome-wide alignment or wet-lab validation.

**Research question:** can position-aware mismatch features prioritize candidates with nonzero SITE-seq cleavage signal while keeping intended-guide groups isolated across folds?

## Why this is the right transfer

The transcript notebook's strongest ideas transfer as methodology rather than as a claim that transcript coding status predicts editing: sequence-derived features, group-aware validation, held-out evaluation, sensitivity to biological grouping, and explicit limitations. For Cas9, the relevant unit is a guide–candidate-site pair, and the grouping key is the intended guide.

The refined ranker adds an **extended PAM-proximal mismatch feature**. Its window start and mismatch weight are selected inside inner grouped folds, then evaluated once on outer held-out guide groups. This is a small, interpretable feature-engineering change rather than a claim that a larger black-box model is automatically better.


In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
if not (ROOT / 'crispr' / 'run_offtarget_study.py').exists():
    # In Colab, clone the public repository first.
    !git clone https://github.com/minagayid/flyrank.git
    ROOT = Path('flyrank')
%cd {ROOT}
!pip -q install -r requirements.txt


In [ ]:
!python3 crispr/run_offtarget_study.py


In [ ]:
results = json.loads(Path('outputs/crispr/study_results.json').read_text())
for key in ['rows','guides','activity_rows','activity_baseline','activity_model','activity_random_forest_comparison','activity_model_selection']:
    print(key, results[key])


## Interpretation

The main task is continuous ranking of assay activity. The original mismatch/seed score remains the control. The refined ranker tunes only the start and weight of an extended PAM-proximal mismatch window inside the training folds.

The improvement is intentionally reported as a small, benchmark-specific change. It does not establish universal Cas9 specificity, and the random-forest comparison remains visible rather than being silently discarded.

The ON/OFF classification metric is a sanity check only: exact intended-guide rows make that task structurally easy. It must not be presented as general Cas9 specificity performance.

## Operational guardrails

1. Use the correct genome assembly and a validated aligner for candidate-site discovery.
2. Treat this model as a triage/ranking aid, not a guide approval system.
3. Check guide diversity, PAM validity, chromatin/cell context, and assay transfer before interpreting a score.
4. Confirm important candidates experimentally and document the reference, software versions, and thresholds.
